# Lesson 10: Site Suitability (Capstone)

**Planning question: Where should a new park go?**

Planners rarely decide on one factor alone. A good park site might be one that is:

- in a **hot** area, where cooling is needed
- in a neighbourhood with **little green cover**
- on **flat** land that is cheap to develop
- close to a **main road**, so people can reach it easily
- currently **open land**, not buildings, water or existing forest

**Suitability analysis** turns each factor into a map scored from 0 to 1, gives each map a weight, and adds them up.
This notebook has two parts:

- **Part 1** brings together the skills from all the earlier lessons to find the best sites.
- **Part 2** introduces **interactive web maps**: you will turn the results into a map that anyone can open, zoom and click in a web browser.

Study area: south-east Bengaluru (HSR Layout, Bellandur, Sarjapur Road).

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q pystac-client planetary-computer odc-stac rioxarray xarray-spatial osmnx

In [ ]:
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import osmnx as ox
import xrspatial
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import Fullscreen, MeasureControl
from rasterio.features import rasterize
from shapely.geometry import shape, box

## Settings

The **weights** say how important each factor is. They must add up to 1.
Changing them is how a planner tests different priorities. Try it at the end.

In [ ]:
BBOX = [77.60, 12.88, 77.76, 13.00]   # west, south, east, north
UTM_CRS = "EPSG:32643"
RESOLUTION = 30                        # metres (the Landsat pixel size)

WEIGHTS = {
    "heat": 0.30,        # hotter = more need for a park
    "green_need": 0.30,  # less green nearby = more need
    "flatness": 0.20,    # flatter = easier to build
    "access": 0.20,      # nearer a main road = easier to reach
}
print("Weights add up to:", sum(WEIGHTS.values()))

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Part 1: The suitability analysis

## Step 1: Heat and green cover from Landsat (Lessons 7 and 8)

We load one clear Landsat image from the hot season. Its 30 m grid becomes the **common grid** for all our layers.

In [ ]:
items = catalog.search(
    collections=["landsat-c2-l2"], bbox=BBOX, datetime="2024-03-01/2024-05-31",
    query={"eo:cloud_cover": {"lt": 10}, "platform": {"in": ["landsat-8", "landsat-9"]}},
).item_collection()
area = box(*BBOX)
item = min(items, key=lambda i: (-round(shape(i.geometry).intersection(area).area / area.area, 2),
                                 i.properties["eo:cloud_cover"]))
print("Using:", item.id, item.datetime.date())

landsat = odc.stac.load([item], bands=["lwir11", "red", "nir08", "qa_pixel"], bbox=BBOX,
                        crs=UTM_CRS, resolution=RESOLUTION).isel(time=0)
grid = landsat.odc.geobox   # the common grid

clear = ((landsat["qa_pixel"] & 0b11010) == 0) & (landsat["lwir11"] > 0)
lst = (landsat["lwir11"].astype("float32") * 0.00341802 + 149.0 - 273.15).where(clear)
red = (landsat["red"].astype("float32") * 0.0000275 - 0.2).where(clear)
nir = (landsat["nir08"].astype("float32") * 0.0000275 - 0.2).where(clear)
ndvi = (nir - red) / (nir + red)

## Step 2: Slope from an elevation model

The **Copernicus DEM** gives the ground height every 30 m. `xrspatial.slope` turns height into steepness in degrees.

In [ ]:
dem_items = catalog.search(collections=["cop-dem-glo-30"], bbox=BBOX).item_collection()
dem = odc.stac.load(dem_items, bands=["data"], geobox=grid, resampling="bilinear")["data"].isel(time=0)
slope = xrspatial.slope(dem.astype("float32"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dem.plot(ax=axes[0], cmap="terrain", cbar_kwargs={"label": "Elevation (m)"})
slope.plot(ax=axes[1], cmap="viridis", vmax=10, cbar_kwargs={"label": "Slope (degrees)"})
axes[0].set_title("Elevation")
axes[1].set_title("Slope")
plt.show()

## Step 3: Where can a park go? Land cover from ESA WorldCover

We only allow sites on **open land**: shrubs (20), grassland (30), cropland (40) and bare ground (60).
Built-up land (50), water (80) and tree cover (10) are left out.

In [ ]:
wc_items = catalog.search(collections=["esa-worldcover"], bbox=BBOX,
                          datetime="2021-01-01/2021-12-31").item_collection()
worldcover = odc.stac.load(wc_items, bands=["map"], geobox=grid, resampling="nearest",
                           groupby="solar_day")["map"].isel(time=0)
open_land = worldcover.isin([20, 30, 40, 60])

open_land.plot(size=6, cmap="Greys", add_colorbar=False)
plt.title(f"Open land available: {float(open_land.mean()) * 100:.1f}% of the area (white)")
plt.show()

## Step 4: Distance to main roads from OpenStreetMap (Lessons 2 and 3)

We download primary, secondary and tertiary roads, burn them into the common grid,
and then use `xrspatial.proximity` to work out every pixel's distance to the nearest road.

In [ ]:
roads = ox.features_from_bbox(bbox=tuple(BBOX),
                              tags={"highway": ["trunk", "primary", "secondary", "tertiary"]})
roads = roads[roads.geom_type.isin(["LineString", "MultiLineString"])].to_crs(UTM_CRS)
print("Road segments:", len(roads))

road_grid = rasterize(((geom, 1) for geom in roads.geometry), out_shape=lst.shape,
                      transform=lst.rio.transform(), fill=0, all_touched=True, dtype="uint8")
road_raster = lst.copy(data=road_grid.astype("float32"))
road_distance = xrspatial.proximity(road_raster, target_values=[1])

road_distance.plot(size=6, cmap="Blues_r", vmax=2000, cbar_kwargs={"label": "Distance to main road (m)"})
plt.title("Distance to main roads")
plt.show()

## Step 5: Turn each factor into a score from 0 to 1

Factors come in different units: degrees C, NDVI, degrees of slope and metres. To combine them, we **rescale** each one to 0 (worst) to 1 (best).

- **heat**: hotter is better (more need). We stretch between the 2nd and 98th percentile so a few extreme pixels do not dominate.
- **green_need**: we average NDVI over the surrounding 500 m (17 by 17 pixels), then flip it so that less green means a higher score.
- **flatness**: 0 degrees scores 1, and 10 degrees or more scores 0.
- **access**: on a road scores 1, and 1 km or more away scores 0.

In [ ]:
def rescale(da):
    low, high = float(da.quantile(0.02)), float(da.quantile(0.98))
    return ((da - low) / (high - low)).clip(0, 1)


neighbourhood_ndvi = ndvi.rolling(x=17, y=17, center=True, min_periods=1).mean()

criteria = {
    "heat": rescale(lst),
    "green_need": 1 - rescale(neighbourhood_ndvi),
    "flatness": 1 - slope.clip(0, 10) / 10,
    "access": 1 - road_distance.clip(0, 1000) / 1000,
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (name, layer) in zip(axes, criteria.items()):
    layer.plot(ax=ax, cmap="RdYlGn", vmin=0, vmax=1, add_colorbar=False)
    ax.set_title(f"{name} (weight {WEIGHTS[name]})")
    ax.set_axis_off()
plt.show()

## Step 6: The weighted overlay

Suitability = (weight x score), added up over all four factors, and kept only on open land.

In [ ]:
suitability = sum(WEIGHTS[name] * criteria[name] for name in WEIGHTS)
suitability = suitability.where(open_land)

fig, ax = plt.subplots(figsize=(10, 8))
suitability.plot(ax=ax, cmap="RdYlGn", vmin=0.2, vmax=0.8, cbar_kwargs={"label": "Suitability score"})
roads.plot(ax=ax, color="black", linewidth=0.3)
ax.set_title("Suitability for a new park (open land only)")
ax.set_axis_off()
plt.show()

## Step 7: Pick the top 10 candidate sites

A single good pixel is too small for a park. We group pixels into blocks of 10 by 10 (300 m by 300 m, about 9 hectares)
and score each block by its average suitability, counting built-up and other excluded land as 0.
This favours blocks that are both suitable and largely open.

In [ ]:
blocks = suitability.fillna(0).coarsen(x=10, y=10, boundary="trim").mean()
top = blocks.to_series().nlargest(10)

half = RESOLUTION * 10 / 2
sites = gpd.GeoDataFrame(
    {"rank": range(1, 11), "score": top.values.round(3)},
    geometry=[box(x - half, y - half, x + half, y + half) for y, x in top.index],
    crs=UTM_CRS,
)
sites

## Step 8: Save the results

In [ ]:
suitability.rio.to_raster("park_suitability.tif")
sites.to_file("park_candidate_sites.gpkg", driver="GPKG")
print("Saved park_suitability.tif and park_candidate_sites.gpkg")

---

# Part 2: Build an interactive web map

Every map in this course so far has been a **static** picture. To share results with decision makers and citizens,
you want a map that people can explore themselves: zoom in, switch layers on and off, and click a site to read about it.
That is an **interactive web map**.

We use **folium**, a Python package that writes **Leaflet** maps. Leaflet is the free JavaScript library behind many
maps on the web. We build the map one layer at a time, so you can see what each step adds.

## Step 9: A base map with two backgrounds

A web map starts with **tiles**: small square images of a background map that load as you zoom and pan.
We add two backgrounds, a light street map and satellite imagery. `tiles=None` starts with an empty map, so we choose both ourselves.

In [ ]:
center = [(BBOX[1] + BBOX[3]) / 2, (BBOX[0] + BBOX[2]) / 2]   # [latitude, longitude]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer("CartoDB positron", name="Street map").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite",
).add_to(m)
m

## Step 10: Add the suitability raster

Web maps use latitude and longitude (EPSG:4326), so we first **reproject** the suitability grid from UTM.
Then we turn the scores into colours with the same colour scale as the static map, make empty pixels transparent,
and lay the picture over the map as an **image overlay**.

In [ ]:
suitability_ll = suitability.rio.write_crs(UTM_CRS).rio.write_nodata(np.nan).rio.reproject("EPSG:4326")
values = suitability_ll.values

colours = plt.get_cmap("RdYlGn")(np.clip((values - 0.2) / 0.6, 0, 1))   # same 0.2 to 0.8 range as Step 6
colours[..., 3] = np.where(np.isnan(values), 0, 1)                    # transparent where there is no score

west, south, east, north = suitability_ll.rio.bounds()
folium.raster_layers.ImageOverlay(
    image=colours,
    bounds=[[south, west], [north, east]],
    name="Park suitability",
    opacity=0.7,
    mercator_project=True,
).add_to(m)
m

## Step 11: Add the main roads

Vector layers go onto the map as **GeoJSON**, the standard format for shapes on the web.
A **tooltip** is the label that appears when you hover over a feature.

In [ ]:
roads_ll = roads[["highway", "geometry"]].to_crs("EPSG:4326")

folium.GeoJson(
    roads_ll,
    name="Main roads",
    style_function=lambda feature: {"color": "#444444", "weight": 1.5},
    tooltip=folium.GeoJsonTooltip(fields=["highway"], aliases=["Road type:"]),
).add_to(m)
m

## Step 12: Add the candidate sites with popups

A **popup** opens when you click a feature. It can hold any information, here the rank, score and size of each site.

In [ ]:
sites["area_ha"] = (sites.area / 10_000).round(1)
sites_ll = sites.to_crs("EPSG:4326")

folium.GeoJson(
    sites_ll,
    name="Candidate park sites",
    style_function=lambda feature: {"color": "yellow", "weight": 3, "fillOpacity": 0.15},
    tooltip=folium.GeoJsonTooltip(fields=["rank"], aliases=["Candidate site:"]),
    popup=folium.GeoJsonPopup(fields=["rank", "score", "area_ha"],
                              aliases=["Rank", "Suitability score", "Area (hectares)"]),
).add_to(m)
m

## Step 13: Add map controls

- **Layer control**: switch backgrounds and layers on and off (always add it last, after all the layers)
- **Full screen** button
- **Measure** tool: click on the map to measure distances and areas, which is handy for checking a site's size

In [ ]:
folium.LayerControl(collapsed=False).add_to(m)
Fullscreen().add_to(m)
MeasureControl(primary_length_unit="meters", primary_area_unit="hectares").add_to(m)
m

Now use the map to **check the model with your own eyes**. Switch to the Satellite background, turn off the suitability layer
and click each candidate site. Is the land really vacant? Is it a lake bed, a construction site or a private campus?
A model is only as good as its data.

## Step 14: Save it as a web page

`m.save()` writes the whole map into **one HTML file**. Anyone can open it in a web browser with no software and no login.
Download it from the **Files** panel on the left (right-click, then **Download**).

To share it online for free:

- **GitHub Pages**: upload the file to a GitHub repository and turn on Pages in the repository settings
- **Netlify Drop** (app.netlify.com/drop): drag the file onto the page to get a public link

In [ ]:
m.save("park_sites_web_map.html")
print("Saved park_sites_web_map.html")

## What this means for planning

- Suitability analysis makes the reasons behind a site choice clear and open to discussion.
  Anyone can see the factors and the weights.
- The weights are a **value judgement**, not a fact. Agree on them with stakeholders, and test how much the answer changes when they change.
- The model gives a shortlist, not a decision. Ownership, cost, master plan zoning and community needs must come next.
- An interactive web map is a strong tool for public consultation: people can find their own street, see the proposed sites and react to them.

## Course wrap-up

You have used OpenStreetMap, street networks, building footprints, Sentinel-2, Landsat, a DEM, a land cover map and machine learning,
and published the result as an interactive web map, all with free data and short Python notebooks.

**Thank you for learning with us.** If the course helped you, please [give it a star on GitHub](https://github.com/soniadas123/rs-gis-spatial-planner) and share it with a colleague or a student. Stars help other planners find it. The same building blocks can answer many other planning questions.

## Try it yourself

1. Set `"heat": 0.6` and lower the other weights so the total is still 1. Do the top sites move?
2. Add a fifth factor: distance to the nearest existing park, using `tags={"leisure": "park"}` and the same proximity method as roads.
3. Change the task: find sites for **affordable housing**. Which factors and weights would you use instead?
4. Web map: add the heat layer (`criteria["heat"]`) as a second image overlay, using the `"inferno"` colour scale.
5. Web map: add the BBMP ward boundaries from Lesson 1 as a `folium.GeoJson` layer, with the ward name (`KGISWardName`) in the tooltip.